# Smoke Test MCMC Standard vs NE-MCMC (2^4, bs=4, beta=6)

Questo notebook testa 4 casi:
1. MCMC standard + azione normale
2. MCMC standard + azione con difetto
3. NE-MCMC + azione normale
4. NE-MCMC + azione con difetto


In [ ]:
import sys
from pathlib import Path
import torch

root = Path.cwd()
if (root / 'src').exists():
    project_root = root
elif (root.parent / 'src').exists():
    project_root = root.parent
else:
    raise RuntimeError('Non trovo la cartella src')

sys.path.append(str(project_root / 'src'))

from neoqcd.utils import create_mask, Defect
from neoqcd.flow import FlowPars
from neoqcd.theory import PriorSUN
from neoqcd.mcmc import NEMCMC_update

torch.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

D = 4
T = 2
L = 2
N = 3
batch_size = 4
beta = 6.0
therm_steps = 1000
mcmc_steps = 3
defect_par = 0.5

print(f'device={device}, lattice={T}x{L}x{L}x{L}, bs={batch_size}, beta={beta}')


device=cpu, lattice=2x2x2x2, bs=4, beta=6.0


In [2]:
def build_flow_pars(defect=None):
    mask = create_mask(D, T, L).to(device)
    flow_pars = FlowPars(
        D=D,
        T=T,
        L=L,
        N=N,
        mask=mask,
        protocol=[beta],
        batch_size=batch_size,
        device=device,
        orsteps=1,
        updates_per_layer=1,
        defect=defect,
    )
    return flow_pars

def run_standard_mcmc(use_defect=False):
    if use_defect:
        defect = Defect(D=D, T=T, L=L, dsize=1, time_slice=0, space_slice=0)
        defect_mask = defect.create_defect_mask().to(device)
        flow_pars = build_flow_pars(defect=defect)
        dpar = defect_par
    else:
        defect_mask = None
        flow_pars = build_flow_pars(defect=None)
        dpar = 1.0

    prior = PriorSUN(
        flow_pars=flow_pars,
        beta=beta,
        therm_steps=therm_steps,
        mcmc_steps=mcmc_steps,
        defect_mask=defect_mask,
        defect_par=dpar,
    )

    cfgs_therm, action_therm = prior()
    cfgs_after, action_after = prior(cfgs_therm.clone())

    assert cfgs_therm.shape[0] == batch_size
    assert torch.isfinite(action_therm).all()
    assert torch.isfinite(action_after).all()

    return {
        'mode': 'defect' if use_defect else 'normal',
        'action_therm_mean': float(action_therm.mean().item()),
        'action_after_mean': float(action_after.mean().item()),
        'delta_action_mean': float((action_after - action_therm).mean().item()),
    }

def run_nemcmc(use_defect=False):
    if use_defect:
        defect = Defect(D=D, T=T, L=L, dsize=1, time_slice=0, space_slice=0)
        defect_mask = defect.create_defect_mask().to(device)
        flow_pars = build_flow_pars(defect=defect)
        dpar = defect_par
    else:
        defect_mask = None
        flow_pars = build_flow_pars(defect=None)
        dpar = 1.0

    prior = PriorSUN(
        flow_pars=flow_pars,
        beta=beta,
        therm_steps=therm_steps,
        mcmc_steps=1,
        defect_mask=defect_mask,
        defect_par=dpar,
    )
    cfgs_start, _ = prior()

    updater = NEMCMC_update(
        flow_pars=flow_pars,
        beta=beta,
        defect_mask=defect_mask,
        defect_par=dpar,
    )

    s_old = updater.action(cfgs_start, defect_mask)
    cfgs_new, dS, _ = updater(cfgs_start.clone(), flow_pars)
    s_new = updater.action(cfgs_new, defect_mask)

    consistency_err = torch.max(torch.abs((s_new - s_old) - dS)).item()

    assert cfgs_new.shape[0] == batch_size
    assert torch.isfinite(s_old).all()
    assert torch.isfinite(s_new).all()
    assert torch.isfinite(dS).all()

    return {
        'mode': 'defect' if use_defect else 'normal',
        's_old_mean': float(s_old.mean().item()),
        's_new_mean': float(s_new.mean().item()),
        'dS_mean': float(dS.mean().item()),
        'consistency_err_max': float(consistency_err),
    }


In [3]:
std_normal = run_standard_mcmc(use_defect=False)
std_normal


{'mode': 'normal',
 'action_therm_mean': 214.8952838159287,
 'action_after_mean': 228.0854677412331,
 'delta_action_mean': 13.190183925304368}

In [4]:
std_defect = run_standard_mcmc(use_defect=True)
std_defect


{'mode': 'defect',
 'action_therm_mean': 235.5318684856993,
 'action_after_mean': 239.38871615020804,
 'delta_action_mean': 3.8568476645087415}

In [5]:
nem_normal = run_nemcmc(use_defect=False)
nem_normal


{'mode': 'normal',
 's_old_mean': 236.72791705288466,
 's_new_mean': 229.251046465755,
 'dS_mean': -7.476870587129689,
 'consistency_err_max': 0.0}

In [6]:
nem_defect = run_nemcmc(use_defect=True)
nem_defect


{'mode': 'defect',
 's_old_mean': 225.89279703104728,
 's_new_mean': 239.67445579665332,
 'dS_mean': 13.781658765606053,
 'consistency_err_max': 0.0}

In [7]:
summary = {
    'standard_normal': std_normal,
    'standard_defect': std_defect,
    'nemcmc_normal': nem_normal,
    'nemcmc_defect': nem_defect,
}
summary


{'standard_normal': {'mode': 'normal',
  'action_therm_mean': 214.8952838159287,
  'action_after_mean': 228.0854677412331,
  'delta_action_mean': 13.190183925304368},
 'standard_defect': {'mode': 'defect',
  'action_therm_mean': 235.5318684856993,
  'action_after_mean': 239.38871615020804,
  'delta_action_mean': 3.8568476645087415},
 'nemcmc_normal': {'mode': 'normal',
  's_old_mean': 236.72791705288466,
  's_new_mean': 229.251046465755,
  'dS_mean': -7.476870587129689,
  'consistency_err_max': 0.0},
 'nemcmc_defect': {'mode': 'defect',
  's_old_mean': 225.89279703104728,
  's_new_mean': 239.67445579665332,
  'dS_mean': 13.781658765606053,
  'consistency_err_max': 0.0}}